In [56]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

In [57]:
fin_txn_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\bronze\finance_transactions.csv")
fraud_pattern_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\bronze\fraud_patterns.csv")
merchant_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\bronze\merchant_db.csv")
user_df = pd.read_csv(r"C:\Users\Rudra\Desktop\smart-transaction-ledger\data\bronze\user_db.csv")

In [58]:
user_df.sample(5)

,user_id,income_range,city,spending_limit,risk_score
19,U020,70000-90000,BBSR,50000.0,0.5
29,U030,50000-70000,Bhubaneswar,10000.0,0.3
13,U014,NaN,NaN,20000.0,0.5
5,U006,70000-90000,BBSR,NaN,high
22,U023,50000-70000,BBSR,10000.0,medium


## 1. City

- Railway Station Name 
- Forward Filling

In [59]:
user_df['city'] = user_df['city'].replace({"Bhubaneswar": 'BSSR', "Bhubaneshwar": 'BSSR', 'Cuttack': 'CTC'})

In [60]:
user_df['city'] = user_df['city'].ffill()

## 2. Income

- Nan value treat as the under 45k.
- Mapping

In [61]:
user_df['income_range'].value_counts()

income_range
>100000        10
50000-70000    10
70000-90000     8
45000-55000     6
30000           5
Name: count, dtype: int64

In [62]:
user_df['income_range_clean'] = user_df['income_range'].fillna('30000')

In [63]:
income_mapping = {
    '>100000': 'Over 100k',
    '70000-90000': '70k - 90k',
    '50000-70000': '50k - 70k',
    '45000-55000': '45k - 55k',
    '30000': 'Under 45k'
}

user_df['income_range_clean'] = user_df['income_range'].map(income_mapping)
user_df['income_range_clean'] = user_df['income_range_clean'].fillna('Under 45k')

## 3. risk score
- Map the risk score
- Low means 0.2, medium is 0.3 and high is 0.5

In [64]:
user_df['risk_score'].unique()

array(['low', '0.2', 'high', '0.5', '0.1', 'medium', nan, '0.3'],
      dtype=object)

In [65]:
user_df['risk_score'].value_counts()

risk_score
0.5       10
low        9
0.2        8
high       8
medium     6
0.3        4
0.1        3
Name: count, dtype: int64

In [66]:
risk_mapping = {
    'low': 0.1,
    'medium': 0.3,
    'high': 0.5
}

user_df['risk_score_new'] = user_df['risk_score'].map(risk_mapping)

In [67]:
user_df['risk_score_new'] = user_df['risk_score_new'].bfill()

## 4. Spending

In [69]:
user_df['spending_limit'].value_counts()

spending_limit
10000.0    10
50000.0    10
15000.0     8
25000.0     8
20000.0     5
Name: count, dtype: int64

In [72]:
user_df['spending_limit'] = user_df['spending_limit'].fillna(user_df['spending_limit'].min())

## Remove Old

In [75]:
user_df

,user_id,income_range,city,spending_limit,risk_score,income_range_clean,risk_score_new
0,U001,NaN,BSSR,10000.0,low,Under 45k,0.1
1,U002,>100000,BBSR,15000.0,0.2,Over 100k,0.5
2,U003,NaN,BBSR,10000.0,high,Under 45k,0.5
3,U004,50000-70000,BSSR,10000.0,0.5,50k - 70k,0.5
4,U005,>100000,BSSR,50000.0,0.1,Over 100k,0.5
5,U006,70000-90000,BBSR,10000.0,high,70k - 90k,0.5
6,U007,>100000,CTC,50000.0,low,Over 100k,0.1
7,U008,50000-70000,BBSR,10000.0,high,50k - 70k,0.5
8,U009,30000,BSSR,15000.0,0.5,Under 45k,0.5
9,U010,30000,BSSR,10000.0,high,Under 45k,0.5


In [77]:
user_df = user_df.drop(columns={'income_range', 'risk_score'})

In [78]:
user_df

,user_id,city,spending_limit,income_range_clean,risk_score_new
0,U001,BSSR,10000.0,Under 45k,0.1
1,U002,BBSR,15000.0,Over 100k,0.5
2,U003,BBSR,10000.0,Under 45k,0.5
3,U004,BSSR,10000.0,50k - 70k,0.5
4,U005,BSSR,50000.0,Over 100k,0.5
5,U006,BBSR,10000.0,70k - 90k,0.5
6,U007,CTC,50000.0,Over 100k,0.1
7,U008,BBSR,10000.0,50k - 70k,0.5
8,U009,BSSR,15000.0,Under 45k,0.5
9,U010,BSSR,10000.0,Under 45k,0.5
